# Premier CNN sur des chiffres 8×8

**Objectifs.** Lire les dimensions image/canal, calculer une taille de convolution, entraîner un petit CNN et inspecter les erreurs. Durée : 90 min. Dépendances optionnelles : `.[dl]`. Données : digits intégrés à scikit-learn, pas de téléchargement.

## 1. Représentation et séparation

Les images niveaux de gris sont de forme `(n, 8, 8)` ; PyTorch Conv2d attend `(batch, channels, height, width)`, donc `(n,1,8,8)`. Normaliser les pixels par 16, valeur maximale de ce dataset.

In [ ]:
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

torch.set_num_threads(1)
torch.manual_seed(10)
digits = load_digits()
X = torch.tensor(digits.images[:, None, :, :] / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=10)
train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
print('images:', tuple(X.shape), '| labels:', tuple(y.shape))
pixel_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
pixel_model.fit(x_train.numpy().reshape(len(x_train), -1), y_train.numpy())
pixel_accuracy = pixel_model.score(x_test.numpy().reshape(len(x_test), -1), y_test.numpy())
print('pixel logistic-regression baseline accuracy:', round(pixel_accuracy, 3))

## 2. Dimensions de la convolution

Avec `kernel=3`, `padding=1`, `stride=1`, la formule retourne `floor((8+2-3)/1)+1=8`. Une couche de 8 filtres a la forme de sortie `(batch,8,8,8)`. Pooling 2×2 réduit chaque dimension spatiale par deux.

In [ ]:
model = nn.Sequential(
    nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(16 * 2 * 2, 10)
)
print('logits shape:', tuple(model(x_train[:5]).shape))

## 3. Entraîner et évaluer

`CrossEntropyLoss` attend des logits bruts de forme `(batch, 10)` et des labels entiers. Softmax est incorporé à la loss ; ne pas appliquer une softmax avant celle-ci.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(6):
    model.train(); total_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        logits = model(images)
        loss = loss_fn(logits, labels)
        loss.backward(); optimizer.step()
        total_loss += loss.item() * len(labels)
    print(f'epoch {epoch + 1}: mean train loss={total_loss / len(train_loader.dataset):.3f}')
model.eval()
with torch.inference_mode():
    test_logits = model(x_test)
    test_prediction = test_logits.argmax(dim=1)
    test_accuracy = (test_prediction == y_test).float().mean().item()
print('held-out accuracy:', round(test_accuracy, 3))

## 4. Inspecter des erreurs

Une moyenne ne montre pas quelles classes se confondent. Afficher quelques images mal classées aide à vérifier les entrées et à formuler des hypothèses.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

errors = torch.where(test_prediction != y_test)[0][:6]
fig, axes = plt.subplots(1, max(1, len(errors)), figsize=(12, 2.5))
for ax, index in zip(np.atleast_1d(axes), errors):
    ax.imshow(x_test[index, 0], cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'true={y_test[index].item()}, pred={test_prediction[index].item()}'); ax.axis('off')
plt.tight_layout(); plt.show()
ConfusionMatrixDisplay(confusion_matrix(y_test.numpy(), test_prediction.numpy())).plot(cmap='Blues')
plt.show()

## 5. Interprétation et limites

Le dataset est petit, monochrome, centré et basse résolution ; ce score n’est pas transférable à des photos du monde réel. Pour continuer : comparer à une régression logistique sur pixels, changer une seule augmentation et vérifier quelles classes gagnent/perdent. Transfer learning, Grad-CAM et segmentation ne sont pas couverts par ce notebook d’introduction.